# HumanAI Detect - Leave-One-Generator-Out (LOGO) Degerlendirme (Colab)

**Amac:** Mevcut held-out set (`make_holdout_split.py`) sadece 'hic gorulmemis DOKUMAN' olcuyor -- ureticiler (Qwen/GPT-4o-mini/Claude) held-out'ta da egitimde de ayni sekilde temsil ediliyor, yani sayilar (Accuracy=0.9100, Macro-F1=0.9191) in-distribution genelleme olcuyor, 'hic gorulmemis ureticiye genelleme' degil. n=15/15 canli testte bunun BUYUK farkli ciktigi zaten kanitlandi (GPT-4o-mini 15/15, Claude 1/15) -- bu notebook o testi otomatik/tekrarlanabilir/buyuk-orneklemli hale getirir: sirayla her ureticiyi (qwen/gpt4o_mini/claude_sonnet5) EGITIMDEN TAMAMEN cikarir, sadece o ureticide test eder.

**Sure tahmini:** Her fold tam bir stacking modeli (4 base model + kendi ic 5-fold'u) egitir -- 3 uretici icin ~3x `measure_calibration.py` tek kosusu kadar surer. Bu yuzden Colab'da (kesintisiz, guclu CPU/RAM) calistiriliyor.

**ONEMLI -- Runtime tipi: CPU + Yuksek RAM secilmeli, GPU DEGIL.** Bu notebook'taki egitim (stacking: XGBoost/CatBoost/MLP/LogReg) GPU'ya ozel yapilandirilmadi -- GPU runtime'i secmek sadece daha az vCPU/RAM ayrilmasina yol acip yavaslatir (bkz. proje notlari, `colab_measure_calibration.ipynb`'de de ayni sebeple CPU+Yuksek RAM kullaniliyor). Runtime > Change runtime type > Hardware accelerator: None, Runtime shape: High-RAM.

**Girdi:** `data/processed/fused.parquet` (yerelden yuklenecek).
**Cikti:** `outputs/reports/cv_results/logo_cv.json` + `.md` (indirilecek).

In [ ]:
# Repo klonla (ilk kez) veya guncelle (klasor zaten varsa git pull)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
import os
if os.path.exists('/content/humanai_detect'):
    %cd /content/humanai_detect
    !git pull origin master
else:
    !git clone {GITHUB_REPO} /content/humanai_detect
    %cd /content/humanai_detect
!git log --oneline -5


In [ ]:
# Bagimliliklari kur (birkac dakika surebilir)
!pip install -e '.[dev]' -q


In [ ]:
# fused.parquet'i yerel makineden yukle
import os
os.makedirs('data/processed', exist_ok=True)
from google.colab import files
uploaded = files.upload()  # data/processed/fused.parquet secilecek
import shutil
for fname in uploaded:
    shutil.move(fname, f'data/processed/{fname}')
print(os.listdir('data/processed'))


In [ ]:
# LOGO-CV calistir (asil is) -- ilerleme her uretici fold'u bitince yazdirilir
!python scripts/logo_cv.py


In [ ]:
# Sonucu ekranda goster
print(open('outputs/reports/cv_results/logo_cv.md', encoding='utf-8').read())


In [ ]:
# Sonuclari yerel makineye indir
from google.colab import files
files.download('outputs/reports/cv_results/logo_cv.json')
files.download('outputs/reports/cv_results/logo_cv.md')


## Indirilen Dosyalari Yerlestirme

Iki dosyayi (`logo_cv.json`, `logo_cv.md`) indirdikten sonra (genelde `Downloads/` klasorune duser) `outputs/reports/cv_results/` altina yerlestir, Claude'a haber ver. Bu json, `train_generator_balanced.py`/`colab_generator_balanced.ipynb` icin **baseline** olarak kullanilacak -- once bu notebook, sonra o calistirilmali.